# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Output below lists all discovered record sets in the dataset using their `@id`, name, and included fields.

In [ ]:
# List all available record sets and their fields

if metadata.record_sets:
    for record_set in metadata.record_sets:
        print(f"RecordSet: @id={record_set['@id']} | name={record_set.get('name', '')}")
        if 'fields' in record_set:
            for field in record_set['fields']:
                print(f"    Field: @id={field['@id']} | name={field.get('name', '')}")
        print()
else:
    print("Record sets are not directly listed under metadata. Attempting alternative method.")
    # Try dataset._metadata_jsonld parsing (hack, for datasets listing recordSet at higher level)
    # The FAIR^2 dataset seems to have no direct record sets in .record_sets; fall back to any potential `recordSet` top-level
    import requests
    schema_json = requests.get(croissant_url).json()
    record_sets = []
    if 'recordSet' in schema_json:
        # recordSet can be a list or singular
        rs = schema_json['recordSet']
        if isinstance(rs, dict):
            rs = [rs]
        for record_set in rs:
            print(f"RecordSet: @id={record_set.get('@id', '')} | name={record_set.get('name', '')}")
            if 'field' in record_set:
                fields = record_set['field']
                if isinstance(fields, dict):
                    fields = [fields]
                for field in fields:
                    print(f"    Field: @id={field.get('@id', '')} | name={field.get('name', '')}")
else:
    print("No record sets found in this dataset schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s discovered from the previous cell.

In [ ]:
# Extract data from each record set

import requests
from collections.abc import Iterable

# Identify record set IDs
schema_json = requests.get(croissant_url).json()

record_sets_json = []
if 'recordSet' in schema_json:
    rs = schema_json['recordSet']
    if isinstance(rs, dict):
        rs = [rs]
    record_sets_json = rs
elif 'recordSets' in schema_json:
    # Some schemas may use recordSets
    rs = schema_json['recordSets']
    if isinstance(rs, dict):
        rs = [rs]
    record_sets_json = rs

record_set_ids = [item.get('@id') for item in record_sets_json]

# Load all record sets
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet @id {record_set_id}, shape: {df.shape}")
            print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Show head of first DataFrame
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"Showing first rows in DataFrame for RecordSet @id {first_record_set_id}:")
    display(dataframes[first_record_set_id].head())
else:
    print("No record set DataFrames loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# EDA: Only if we successfully found and loaded a usable DataFrame
import numpy as np

if dataframes:
    # Select the first DataFrame for demonstration
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    
    # Try to auto-detect a numeric column
    numeric_field_id = None
    for col in df.columns:
        col_vals = df[col].dropna()
        if col_vals.size > 0:
            # Try converting to float, look for numeric
            try:
                _ = col_vals.astype(float)
                numeric_field_id = col
                break
            except Exception:
                continue
    if numeric_field_id is None:
        print("No numeric field detected.")
    else:
        # Ensure numeric dtype
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = np.nanmedian(df[numeric_field_id])
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records (top 5):")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try to group by a string column (non-numeric)
        group_field = None
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field_id:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field} showing mean {numeric_field_id}:")
            print(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below, we plot the distribution of the numeric field used above, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # If we grouped by a field earlier, show bar plot
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10, 5))
        top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(15)
        sns.barplot(data=top_groups, x=group_field, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field} (top 15)")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion
Summarize key findings and observations from dataset exploration. For example, describe numeric field distributions, size of loaded data, or differentiation between groups if seen above.

_This notebook demonstrated the use of the `mlcroissant` library for programmatically exploring and analyzing a Croissant-based dataset. For more advanced transformations, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/)._